# Heat Pump Targeting
This notebook compares direct and indirect Carnot-based heat pump and refrigeration targets on a real plant case and shows how the target load changes the utility picture.


## Case context
The packaged `chocolate_factory.json` case is a realistic multi-zone plant sample. `PinchWorkspace` keeps named study cases, while each case remains a real `PinchProblem` so advanced workflows still use the normal `target`, `plot`, `summary_frame`, and bundle lifecycle.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from OpenPinch import PinchWorkspace
from OpenPinch.lib.enums import HPRcycle, ProblemTableLabel
from OpenPinch.services.heat_pump_integration.common import (
    plot_multi_hp_profiles_from_results,
)
from OpenPinch.utils import get_value

WORKFLOW_LABELS = {
    "direct_heat_pump": "Direct Heat Pump",
    "direct_refrigeration": "Direct Refrigeration",
    "indirect_heat_pump": "Indirect Heat Pump",
    "indirect_refrigeration": "Indirect Refrigeration",
}


In [2]:
workspace = PinchWorkspace(
    source="chocolate_factory.json",
    project_name="Site",
)
load_fractions = [0.25, 0.5, 0.75, 1.0]

for load_fraction in load_fractions:
    load_case_name = f"load_{int(load_fraction * 100):03d}"
    load_case = workspace.copy_case("baseline", load_case_name, activate=False)
    load_case.update_options(
        {
            "HPR_TYPE": HPRcycle.MultiTempCarnot.value,
            "HPR_LOAD_VALUE": load_fraction,
            "MAX_HP_MULTISTART": 10,
            "N_COND": 3,
            "N_EVAP": 3,
            "REFRIGERANTS": "water;ammonia",
        }
    )
    for workflow_name in WORKFLOW_LABELS:
        workspace.copy_case(
            load_case_name,
            f"{load_case_name}__{workflow_name}",
            activate=False,
        )

{
    "case_count": len(workspace.list_cases()),
    "active_case": workspace.active_case_name,
    "example_cases": workspace.list_cases()[:8],
}


{'case_count': 21,
 'active_case': 'baseline',
 'example_cases': ['baseline',
  'load_025',
  'load_025__direct_heat_pump',
  'load_025__direct_refrigeration',
  'load_025__indirect_heat_pump',
  'load_025__indirect_refrigeration',
  'load_050',
  'load_050__direct_heat_pump']}

## Solve one named workflow case through `PinchWorkspace`
This keeps the study registry on the workspace while the active case still behaves like a normal `PinchProblem`.


In [3]:
direct_hp_full_case = workspace.case("load_100__direct_heat_pump")
direct_hp_full_target = direct_hp_full_case.target.direct_heat_pump()
direct_hp_full_summary = direct_hp_full_case.summary_frame()
direct_hp_full_summary


,Target,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,Site/Direct Integration,1082.53845,3401.694392,440.472508,33.0,33.0,"HPS: 147.64, LPS: 543.48, HTHW: 391.42, CHW: 0.00","CW: 3401.69, CHW: 0.00"
1,Site/Direct Heat Pump,0.00000,0.000000,0.000000,NaN,NaN,,


In [4]:
direct_hp_catalog = direct_hp_full_case.plot.catalog()

{
    "target_name": direct_hp_full_target.name,
    "available_hpr_graphs": sorted(
        direct_hp_catalog.loc[
            direct_hp_catalog["Graph Type"].str.contains("Heat Pump|Site Utility", na=False),
            "Graph Type",
        ].unique()
    ),
}


{'target_name': 'Site/Direct Heat Pump',
 'available_hpr_graphs': ['Grand Composite Curve with Heat Pump']}

## When you need target internals, use the case directly
Because the workspace returns real `PinchProblem` cases, detailed COP checks and HPR stream-profile inspection do not need any helper wrapper.


In [5]:
profile_problem = workspace.case("load_100__direct_heat_pump")
direct_hp_target = profile_problem.target.direct_heat_pump()
direct_hp_zone_target = profile_problem.master_zone.targets["Direct Heat Pump"]

print({"direct_heat_pump_cop": direct_hp_target.hpr_cop})
plot_multi_hp_profiles_from_results(
    T_hot=direct_hp_zone_target.pt.col[ProblemTableLabel.T.value],
    T_cold=direct_hp_zone_target.pt.col[ProblemTableLabel.T.value],
    H_hot=direct_hp_zone_target.pt.col[ProblemTableLabel.H_NET_HOT.value],
    H_cold=direct_hp_zone_target.pt.col[ProblemTableLabel.H_NET_COLD.value],
    hpr_hot_streams=direct_hp_target.hpr_hot_streams,
    hpr_cold_streams=direct_hp_target.hpr_cold_streams,
)


{'direct_heat_pump_cop': 6.554180780166235}


## Compare direct heat-pump load variants through workspace case comparisons
Each load point is a named case, so direct heat-pump deltas can be compared with the default `DataFrame` comparison output.


In [6]:
direct_hp_cases = [
    f"load_{int(load_fraction * 100):03d}__direct_heat_pump"
    for load_fraction in load_fractions
]
for case_name in direct_hp_cases:
    workspace.case(case_name).target.direct_heat_pump()

direct_hp_delta_frame = pd.concat(
    [
        workspace.compare_cases(
            direct_hp_cases[0],
            case_name,
            target_name="Site/Direct Heat Pump",
            base_label=direct_hp_cases[0],
            other_label=case_name,
        ).loc[["Change"]].assign(variant=case_name)
        for case_name in direct_hp_cases[1:]
    ],
    ignore_index=False,
).reset_index(drop=True)
direct_hp_delta_frame


,Target,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,variant
0,Site/Direct Heat Pump,0.0,0.0,0.0,NaN,NaN,load_050__direct_heat_pump
1,Site/Direct Heat Pump,0.0,0.0,0.0,NaN,NaN,load_075__direct_heat_pump
2,Site/Direct Heat Pump,0.0,0.0,0.0,NaN,NaN,load_100__direct_heat_pump


## Workflow comparison by load fraction
The workspace owns the named cases. Each case still runs the underlying heat-pump or refrigeration method directly through the normal `PinchProblem.target` accessor.


In [7]:
rows = []
for load_fraction in load_fractions:
    for workflow_name, workflow_label in WORKFLOW_LABELS.items():
        case_name = f"load_{int(load_fraction * 100):03d}__{workflow_name}"
        problem = workspace.case(case_name)
        target = getattr(problem.target, workflow_name)()
        rows.append(
            {
                "load fraction": load_fraction,
                "workflow": workflow_label,
                "case": case_name,
                "Qh": get_value(getattr(target, "Qh", None)),
                "Qc": get_value(getattr(target, "Qc", None)),
                "Qr": get_value(getattr(target, "Qr", None)),
                "hpr_work": get_value(getattr(target, "hpr_work", None)),
                "hpr_cop": get_value(getattr(target, "hpr_cop", None)),
                "hpr_success": getattr(target, "hpr_success", None),
                "graph_count": len(problem.plot.catalog()),
            }
        )

comparison = pd.DataFrame(rows)
comparison


KeyError: 'Direct Integration'

## Persist the study bundle
Once the workspace holds the named study cases you care about, save the whole study as a portable JSON bundle.


In [ ]:
bundle_workspace = TemporaryDirectory()
bundle_path = Path(bundle_workspace.name) / "chocolate_factory_hpr_workspace.json"
workspace.save_bundle(bundle_path)
reloaded = PinchWorkspace.load_bundle(bundle_path)

{
    "bundle_path": str(bundle_path),
    "case_count": len(reloaded.list_cases()),
    "sample_heat_pump_cases": [
        name for name in reloaded.list_cases() if name.endswith("__direct_heat_pump")
    ][:3],
}


## Interpretation
A useful HPR option should improve the utility picture at a load that is still operationally plausible. In this workflow, `PinchWorkspace` keeps the named study cases and bundle lifecycle together, while each case remains a normal `PinchProblem` for COP checks, stream profiles, and direct workflow execution.
